# Notebook 03 — Spatial Clustering

## Goal
Group wells by hydro-geological similarity to identify regions that share
contamination risk profiles. Clusters feed directly into the **anomaly detection**
step (Notebook 04): a well is flagged as anomalous relative to its cluster peers,
not the global dataset.

### Algorithms
| Method | Why |
|--------|-----|
| **DBSCAN** (primary) | Density-based, automatically finds the number of clusters, handles noise wells naturally — ideal for geospatial data |
| **Agglomerative** (secondary) | Ward linkage hierarchical clustering for comparison and dendrogram insight |

### Quality Metric
**Silhouette Score** — measures how similar each well is to its own cluster
vs. the nearest neighbouring cluster (range: −1 to +1, >0.5 = strong structure).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.clustering import run_clustering_pipeline

PROCESSED_DIR = '../data/processed'
FIGURES_DIR   = '../figures'

## Step 1 — Run Full Clustering Pipeline

In [ ]:
df_clustered, metrics = run_clustering_pipeline(PROCESSED_DIR, figures_dir=FIGURES_DIR)

print('\n=== Clustering Quality Metrics ===')
for k, v in metrics.items():
    print(f'  {k:<30s}: {v}')

## Step 2 — Cluster Distribution

In [ ]:
print('DBSCAN cluster sizes (−1 = noise):')
print(df_clustered['cluster_dbscan'].value_counts().sort_index())

print('\nAgglomerative cluster sizes:')
print(df_clustered['cluster_agg'].value_counts().sort_index())

## Step 3 — Contamination Rate by DBSCAN Cluster

This is the key cross-tabulation: which clusters have the highest contamination prevalence?
These are the clusters we should prioritise for testing in the active learning loop.

In [ ]:
import pandas as pd

summary = (
    df_clustered[df_clustered['cluster_dbscan'] != -1]
    .groupby('cluster_dbscan')
    .agg(
        count=('cluster_dbscan', 'count'),
        contamination_rate=('is_contaminated', 'mean'),
        mean_lat=('latitude', 'mean'),
        mean_lon=('longitude', 'mean'),
        mean_gwl=('groundwater_level_m', 'mean'),
    )
    .sort_values('contamination_rate', ascending=False)
)
summary['contamination_rate'] = summary['contamination_rate'].map('{:.2%}'.format)
print(summary.to_string())

## Step 4 — View Saved Figures

In [ ]:
from IPython.display import Image, display

for fname in [
    'clusters_map_dbscan.png',
    'silhouette_dbscan.png',
    'clusters_map_agglomerative.png',
    'dendrogram_hierarchical.png',
]:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n--- {fname} ---')
        display(Image(fpath))